In [1]:
# # Install Pytorch & other libraries
# %pip install "torch==2.4.1" tensorboard 
# %pip install flash-attn "setuptools<71.0.0" scikit-learn 
 
# # Install Hugging Face libraries
# %pip install  --upgrade \
#   "datasets==3.1.0" \
#   "accelerate==1.2.1" \
#   "hf-transfer==0.1.8"
#   #"transformers==4.47.1" \
 
# # ModernBERT is not yet available in an official release, so we need to install it from github
# %pip install "git+https://github.com/huggingface/transformers.git@6e0515e99c39444caae39472ee1b2fd76ece32f1" --upgrade

In [1]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [2]:
from datasets import load_dataset, concatenate_datasets
 
# Dataset id from huggingface.co/dataset
dataset_id = "youralien/feedback_qesconv_16wayclassification"
dataset_id_care = "youralien/CARE_10percent_16wayclassification"

# Load raw dataset
raw_dataset = load_dataset(dataset_id, split="train") # happens to be called train
care_raw_dataset = load_dataset(dataset_id_care, split="train") # happens to be called train

print(f"FeedbackESConv Raw dataset size: {len(raw_dataset)}")
print(f"CARE raw dataset size: {len(care_raw_dataset)}")

/nlp/scr/rylouie/miniconda3/envs/llama/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


FeedbackESConv Raw dataset size: 8179
CARE raw dataset size: 370


In [3]:
split_dataset = raw_dataset.train_test_split(test_size=0.05, seed=0)
print(f"Train dataset size: {len(split_dataset['train'])}")
print(f"Test dataset size: {len(split_dataset['test'])}")
split_dataset['train'][0]

Train dataset size: 7770
Test dataset size: 409


{'conv_index': 252,
 'helper_index': 9,
 'input': ["Seeker: I am sorry. I can't chat any more please end of this survey. Ok. I can try.",
  'Helper: I am sorry to hear that. I believe in you, things will be better with time. Would you like to continue chatting?',
  'Seeker: Yes',
  'Helper: Okay. Are you excited for the upcoming holidays?',
  'Seeker: Yeah, i am excited upcoming chrisms and new year party.',
  'Helper: That sounds fun! Is the new year party with friends? Or is it a family affair?',
  "Seeker: New year party is with my friend and family affairs. It's very excited and lot of fun and games.",
  'Helper: That is fun! Friends and family are the most wonderful cure for the blues. What types of games will you be playing?'],
 'Reflections-goodareas': 0,
 'Validation-goodareas': 0,
 'Empathy-goodareas': 1,
 'Questions-goodareas': 1,
 'Suggestions-goodareas': 0,
 'Self-disclosure-goodareas': 0,
 'Structure-goodareas': 0,
 'Professionalism-goodareas': 0,
 'Reflections-badareas': 

In [4]:
eval_set = concatenate_datasets([split_dataset['test'], care_raw_dataset])
eval_set

Dataset({
    features: ['conv_index', 'helper_index', 'input', 'Reflections-goodareas', 'Validation-goodareas', 'Empathy-goodareas', 'Questions-goodareas', 'Suggestions-goodareas', 'Self-disclosure-goodareas', 'Structure-goodareas', 'Professionalism-goodareas', 'Reflections-badareas', 'Validation-badareas', 'Empathy-badareas', 'Questions-badareas', 'Suggestions-badareas', 'Self-disclosure-badareas', 'Structure-badareas', 'Professionalism-badareas', 'therapist_id', 'chat_code', 'therapist_index', 'Session Management-goodareas', 'Session Management-badareas'],
    num_rows: 779
})

In [5]:
split_dataset['test'] = eval_set

In [6]:
print(f"Train dataset size: {len(split_dataset['train'])}")
print(f"Test dataset size: {len(split_dataset['test'])}")

Train dataset size: 7770
Test dataset size: 779


### Targeted Sweep of Top Performing RoBERTa Hyperparams with Downsampling + Upweighting Majority Class

In [7]:
import torch
import gc

import evaluate
import numpy as np

def compute_metrics_fn(eval_preds):
    metrics = dict()
    
    accuracy_metric = evaluate.load('accuracy')
    precision_metric = evaluate.load('precision')
    recall_metric = evaluate.load('recall')
    f1_metric = evaluate.load('f1')
    
    logits = eval_preds.predictions
    labels = eval_preds.label_ids
    preds = np.argmax(logits, axis=-1)  
    
    metrics.update(accuracy_metric.compute(predictions=preds, references=labels))
    metrics.update(precision_metric.compute(predictions=preds, references=labels, average='binary'))
    metrics.update(recall_metric.compute(predictions=preds, references=labels, average='binary'))
    metrics.update(f1_metric.compute(predictions=preds, references=labels, average='binary'))

    # Print some predictions
    print(f"Some predictions: {preds[:10]}")
    
    return metrics
    
def cleanup(things_to_delete: list | None = None):
    if things_to_delete is not None:
        for thing in things_to_delete:
            if thing is not None:
                del thing

    gc.collect()
    torch.cuda.empty_cache()


In [8]:
# !pip install autoawq

In [9]:
# from transformers import AutoModelForCausalLM, AutoTokenizer

# model_name = "Qwen/QwQ-32B-AWQ"

# model = AutoModelForCausalLM.from_pretrained(
#     model_name,
#     torch_dtype="float16", # auto
#     device_map="auto"
# )
# tokenizer = AutoTokenizer.from_pretrained(model_name)

# prompt = "How many r's are in the word \"strawberry\""
# messages = [
#     {"role": "user", "content": prompt}
# ]
# text = tokenizer.apply_chat_template(
#     messages,
#     tokenize=False,
#     add_generation_prompt=True
# )

# model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

# generated_ids = model.generate(
#     **model_inputs,
#     max_new_tokens=32768
# )
# generated_ids = [
#     output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
# ]

# response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
# print(response)


We suggest you to set `torch_dtype=torch.float16` for better efficiency with AWQ.
Loading checkpoint shards: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5/5 [00:02<00:00,  1.78it/s]


Okay, so I need to figure out how many times the letter 'r' appears in the word "strawberry". Let me start by writing down the word and looking at each letter one by one. 

First, let's spell out "strawberry" to make sure I have all the letters right. S-T-R-A-W-B-E-R-R-Y. Wait, let me check that again because sometimes I might mix up the letters in "strawberry". Let me count the letters step by step.

Starting with the first letter: S. Then T, then R. So that's the first 'r' right there. Next comes A, W, B, E, and then another R. So that's the second 'r'. Then another R, so that's the third 'r', and finally Y. Hmm, so that's three Rs? Let me verify again because sometimes it's easy to miscount.

Breaking it down letter by letter:

1. S
2. T
3. R (1st R)
4. A
5. W
6. B
7. E
8. R (2nd R)
9. R (3rd R)
10. Y

Yes, so the letters at positions 3, 8, and 9 are all Rs. That makes three Rs in total. Wait, but I might be making a mistake here. Let me think again. The word "strawberry" is spelled

## Run a model on the validation dataset

In [16]:
import torch
import pandas as pd
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from sklearn.metrics import classification_report, confusion_matrix
from tqdm import tqdm
import json
import time
import logging
import os
from datetime import datetime

# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("qwen_evaluation.log"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

class QwenCounselingEvaluator:
    def __init__(
        self, 
        model_name="Qwen/QwQ-32B-AWQ", 
        dtype="float16",
        max_tokens=128,
        temperature=0.1,
        output_dir="qwen_evaluation_results"
    ):
        """
        Initialize the evaluator with a Qwen model.
        
        Args:
            model_name (str): Hugging Face model ID for Qwen
            dtype (str): Model precision type
            max_tokens (int): Maximum tokens to generate for each prediction
            temperature (float): Sampling temperature
            output_dir (str): Directory to save evaluation results
        """
        self.model_name = model_name
        self.max_tokens = max_tokens
        self.temperature = temperature
        self.output_dir = output_dir
        
        # Create output directory if it doesn't exist
        os.makedirs(output_dir, exist_ok=True)
        
        logger.info(f"Initializing QwenCounselingEvaluator with model: {model_name}")
        
        # Load model and tokenizer
        self.dtype = torch.float16 if dtype == "float16" else torch.bfloat16
        try:
            self.model = AutoModelForCausalLM.from_pretrained(
                model_name,
                torch_dtype=self.dtype,
                device_map="auto"
            )
            self.tokenizer = AutoTokenizer.from_pretrained(model_name)
            logger.info(f"Successfully loaded model and tokenizer")
        except Exception as e:
            logger.error(f"Failed to load model: {e}")
            raise
    
    def evaluate_dataset(
        self, 
        dataset, 
        skill_class,
        context_size=5,
        save_predictions=True,
        save_responses=True
    ):
        """
        Evaluate the model on a dataset for a specific counseling skill.
        
        Args:
            dataset: HuggingFace dataset with train/test splits
            skill_class (str): The skill class to evaluate (e.g., "Reflections-goodareas")
            context_size (int): Number of prior messages to include as context
            save_predictions (bool): Whether to save predictions to disk
            save_responses (bool): Whether to save full model responses
            
        Returns:
            dict: Evaluation metrics
        """
        logger.info(f"Starting evaluation for skill: {skill_class}")
        
        # Prepare the dataset
        test_dataset = self._prepare_dataset(dataset, skill_class, context_size)
        
        # Extract labels
        if skill_class in test_dataset.features:
            labels = test_dataset[skill_class]
        else:
            # If the column has been renamed to "labels"
            labels = test_dataset["labels"] if "labels" in test_dataset.features else []
        
        start_time = time.time()
        
        # Process examples in batches
        all_predictions = []
        all_confidences = []
        all_responses = []

        num_errors = 0
        
        for i in tqdm(range(0, len(test_dataset)), desc=f"Evaluating {skill_class}"):

            example = test_dataset[i]

            example = self._prepare_input_text(example, context_size)
            
            # Create prompt for reflection prediction
            prompt = self._create_skill_prompt(example['text'], example['context'], skill_class)
            
            # Generate prediction
            response = self._generate_prediction(prompt)
            
            # Parse prediction
            prediction, confidence = self._parse_prediction(response, skill_class)
                    
            all_predictions.append(prediction)
            all_confidences.append(confidence)
            all_responses.append(response)
        
        elapsed_time = time.time() - start_time
        logger.info(f"Evaluation completed in {elapsed_time:.2f} seconds")
        
        # Compute metrics
        metrics = self._compute_metrics(all_predictions, labels, all_confidences)
        logger.info(f"Evaluation metrics: {metrics}")
        
        # Save results if requested
        if save_predictions or save_responses:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            
            results_data = {
                "example_id": list(range(len(test_dataset))),
                "text": [ex["text"] for ex in test_dataset],
                "true_label": labels if len(labels) > 0 else [None] * len(test_dataset),
                "prediction": all_predictions,
                "confidence": all_confidences
            }
            
            if save_responses:
                results_data["model_response"] = all_responses
            
            # Create DataFrame and save to CSV
            results_df = pd.DataFrame(results_data)
            output_file = f"{self.output_dir}/{skill_class}_{timestamp}.csv"
            results_df.to_csv(output_file, index=False)
            logger.info(f"Saved results to {output_file}")
            
            # Save metrics as JSON
            metrics_file = f"{self.output_dir}/{skill_class}_metrics_{timestamp}.json"
            with open(metrics_file, 'w') as f:
                json.dump(metrics, f, indent=2)
        
        return metrics
    
    def _prepare_dataset(self, dataset, skill_class, context_size):
        """
        Prepare dataset for evaluation.
        
        Args:
            dataset: HuggingFace dataset
            skill_class (str): The skill class to evaluate
            context_size (int): Number of prior messages to include as context
            
        Returns:
            dataset: Processed test dataset
        """
        logger.info("Preparing dataset for evaluation")
        
        # Use the test split
        test_dataset = dataset["test"]
        
        # Check if we need to rename the target column to "labels"
        if skill_class in test_dataset.features and "labels" not in test_dataset.features:
            test_dataset = test_dataset.rename_column(skill_class, "labels")
        
        return test_dataset

    def _prepare_input_text(self, example, context_size=1):
        """
        [-6] Seeker: 
        [-5] Helper:
        [-4] Seeker: 
        [-3] Helper:
        [-2] Seeker: 
        [-1] Helper: Response to classify
        """
        # Convert the last two items of input list to a single text
        response_to_classify = example['input'][-1]
        if context_size is None:
            context = "\n".join(example['input'][:-1])
        else:
            context_start_idx = -1 - context_size
            context = "\n".join(example['input'][context_start_idx:-1])
        return {
            'text': f"{response_to_classify}",
            'context': context,
            **{k:v for k,v in example.items() if k != 'input'}  # Keep other fields
        }
    
    def _create_skill_prompt(self, text, conversation_history=None, skill_class="Reflections"):
        """
        Create a prompt for the model to detect a counseling skill.
        
        Args:
            text (str): The counselor's response text
            conversation_history (str, optional): Previous messages
            skill_class (str): The skill to evaluate
            
        Returns:
            str: Formatted prompt
        """
        # Extract main skill from class name (e.g., "Reflections-goodareas" -> "Reflections")
        skill = skill_class.split('-')[0] if '-' in skill_class else skill_class

        reflection_long_description = """capturing and returning to clients something they have communicated, either explicitly or implicitly stated. A reflection typically mirrors back content from the client's immediately preceding statement, though it may sometimes reference earlier parts of the conversation. 

The key purpose of reflections is to
- Offer the other person’s content back to them in a non-threatening way
- Invite the other person to continue speaking on a topic or delve deeper
- Help the person to organize their thoughts
- Help the person recognize his/her own change talk
*Simple Reflections* simply repeat or rephrase what the person has said. They typically convey understanding or facilitate client/therapist exchanges. These reflections add little or no meaning (or emphasis) to what clients have said. 
*Complex Reflections* typically add substantial meaning or emphasis to what the client has said. These reflections serve the purpose of conveying a deeper or more complex picture of what the client has said. Sometimes the therapist may choose to emphasize a particular part of what the client has said to make a point or take the conversation in a different direction. 

Below are some specific types of areas for improvement in the Reflection technique.
a) Not reflecting, drawing conclusions from the helper’s experience without listening to what the seeker is saying and checking it out with them
b) Making assumptions beyond what was said
c) Copying the seeker's words’ exactly
d) Stating feelings too definitely rather than tentatively (e.g. you obviously feel X vs. I wonder if you feel X)
e) Becoming repetitive, not varying the format of restatements (e.g. I’m hearing you feel sad, I’m hearing you feel anxious, I’m hearing you…)
f) Labeling feelings inaccurately
g) Not capturing the most salient feeling
h) Reflecting on many feelings at the same time
i) Being judgmental
j) Focusing on the feelings of others and not the seeker
k) Reflecting when the seeker is resistant to expressing feelings and reflection might add more pressure"""
        
        # Define skill description
        skill_descriptions = {
            "Reflections": reflection_long_description,
            "Validation": "acknowledging and accepting the client's emotions or experiences as valid",
            "Empathy": "demonstrating understanding of the client's emotional experience",
            "Questions": "asking the client to provide information or explore thoughts and feelings",
            "Suggestions": "offering ideas, advice, or potential solutions to the client",
            "Self-disclosure": "sharing personal information or experiences with the client",
            "Structure": "guiding the session or conversation in a purposeful way",
            "Professionalism": "maintaining ethical boundaries and professional standards"
        }
        
        skill_description = skill_descriptions.get(
            skill, 
            "an important counseling technique that helps clients feel understood"
        )
        
        context_text = f"\nConversation History: \"{conversation_history}\"\n" if conversation_history else ""
        
        prompt = f"""You are an expert in analyzing counseling techniques. Determine if the following counselor response uses the skill of {skill} in an appropriate way.

{skill} involves {skill_description}.

{context_text}
Counselor's response: "{text}"

Analysis instructions:
1. Identify any language in the response that might represent the {skill} technique.
2. Consider whether the counselor's response used this technique in excellent way in, i.e. no significant areas for improvement.
3. Conclude with a clear YES (excellent) or NO (needs improvement) judgment.
4. After your internal thinking, provide ONLY a single-word answer: YES or NO. 

FINAL ANSWER: (YES/NO)"""

        return prompt
    
    def _generate_prediction(self, prompt):
        """
        Generate a prediction from the model.
        
        Args:
            prompt (str): The formatted prompt
            
        Returns:
            str: Model's response
        """
        # Format as chat
        messages = [{"role": "user", "content": prompt}]
        text = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        
        # Tokenize
        model_inputs = self.tokenizer([text], return_tensors="pt").to(self.model.device)
        
        # Generate
        with torch.no_grad():
            generated_ids = self.model.generate(
                **model_inputs,
                max_new_tokens=self.max_tokens,
                temperature=self.temperature,
                do_sample=False  # Deterministic for evaluation
            )
        
        # Extract only the new tokens
        generated_ids = [
            output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
        ]
        
        # Decode
        response = self.tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
        
        return response
    
    def _parse_prediction(self, response, skill_class):
        """
        Parse the model's response to extract the prediction.
        
        Args:
            response (str): The model's response
            skill_class (str): The skill being evaluated
            
        Returns:
            tuple: (prediction, confidence)
                - prediction (int): 1 if skill present, 0 if not
                - confidence (float): Confidence score (0.5-1.0)
        """
        response_lower = response.lower()
        
        # Check for explicit final answer
        if "final answer: yes" in response_lower or "final answer: (yes)" in response_lower:
            return 1, 0.9
        elif "final answer: no" in response_lower or "final answer: (no)" in response_lower:
            return 0, 0.9
            
        # Look for yes/no at the end
        lines = response_lower.split('\n')
        for line in reversed(lines):
            if "yes" in line and "no" not in line:
                return 1, 0.8
            elif "no" in line and "yes" not in line:
                return 0, 0.8
        
        # Count yes/no mentions
        yes_count = response_lower.count(" yes ")
        no_count = response_lower.count(" no ")
        
        if yes_count > no_count:
            confidence = 0.5 + min((yes_count - no_count) * 0.1, 0.3)
            return 1, confidence
        elif no_count > yes_count:
            confidence = 0.5 + min((no_count - yes_count) * 0.1, 0.3)
            return 0, confidence
        else:
            # Default to no with low confidence if unclear
            return 0, 0.5
    
    def _compute_metrics(self, predictions, labels, confidences=None):
        """
        Compute evaluation metrics.
        
        Args:
            predictions (list): Model predictions
            labels (list): Ground truth labels
            confidences (list, optional): Confidence scores for predictions
            
        Returns:
            dict: Evaluation metrics
        """
        if not labels or len(labels) == 0:
            logger.warning("No labels provided for evaluation")
            return {"error": "No labels available for evaluation"}
        
        # Convert to numpy arrays
        y_pred = np.array(predictions)
        y_true = np.array(labels)
        
        # Basic classification report
        report = classification_report(y_true, y_pred, output_dict=True)
        
        # Confusion matrix
        cm = confusion_matrix(y_true, y_pred)
        
        metrics = {
            "accuracy": report["accuracy"],
            "precision": report["1"]["precision"] if 1 in report else 0,
            "recall": report["1"]["recall"] if 1 in report else 0,
            "f1-score": report["1"]["f1-score"] if 1 in report else 0,
            "support": int(report["1"]["support"]) if 1 in report else 0,
            "confusion_matrix": {
                "true_negative": int(cm[0, 0]),
                "false_positive": int(cm[0, 1]),
                "false_negative": int(cm[1, 0]),
                "true_positive": int(cm[1, 1])
            }
        }
        
        # Add confidence metrics if provided
        if confidences:
            conf_array = np.array(confidences)
            metrics["mean_confidence"] = float(np.mean(conf_array))
            metrics["confidence_correct"] = float(np.mean(conf_array[y_pred == y_true]))
            metrics["confidence_incorrect"] = float(np.mean(conf_array[y_pred != y_true])) if np.any(y_pred != y_true) else 0
        
        return metrics
    
    def cleanup(self):
        """Release resources."""
        del self.model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        logger.info("Resources cleaned up")

In [18]:
qwen_evaluator = QwenCounselingEvaluator(max_tokens=8192, output_dir="qwen_Reflections-ann-guidelines_eval_results")

Loading checkpoint shards: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5/5 [00:04<00:00,  1.08it/s]


In [17]:
qwen_evaluator.cleanup()

In [13]:
# Prepare the dataset
test_dataset = qwen_evaluator._prepare_dataset(split_dataset, skill_class="Reflections-goodareas", context_size=5)

# Extract labels
if "Reflections-goodareas" in test_dataset.features:
    labels = test_dataset[skill_class]
else:
    # If the column has been renamed to "labels"
    labels = test_dataset["labels"] if "labels" in test_dataset.features else []

In [19]:
# Get client context if available
example = test_dataset[0]
prepped_example = qwen_evaluator._prepare_input_text(example)

prompt = qwen_evaluator._create_skill_prompt(prepped_example['text'], prepped_example['context'], "Reflections-goodareas")
print(prompt)

You are an expert in analyzing counseling techniques. Determine if the following counselor response contains the skill of Reflections.

Reflections involves capturing and returning to clients something they have communicated, either explicitly or implicitly stated. A reflection typically mirrors back content from the client's immediately preceding statement, though it may sometimes reference earlier parts of the conversation. 

The key purpose of reflections is to
- Offer the other person’s content back to them in a non-threatening way
- Invite the other person to continue speaking on a topic or delve deeper
- Help the person to organize their thoughts
- Help the person recognize his/her own change talk
*Simple Reflections* simply repeat or rephrase what the person has said. They typically convey understanding or facilitate client/therapist exchanges. These reflections add little or no meaning (or emphasis) to what clients have said. 
*Complex Reflections* typically add substantial mea

In [20]:
# Generate prediction
response = qwen_evaluator._generate_prediction(prompt)

/nlp/scr/rylouie/miniconda3/envs/llama/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:629: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.1` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/nlp/scr/rylouie/miniconda3/envs/llama/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:634: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.95` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/nlp/scr/rylouie/miniconda3/envs/llama/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:651: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `40` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


In [21]:
print(response)

Okay, let's tackle this. The user wants to know if the counselor's response uses the Reflections technique properly. First, I need to recall what Reflections entail. From the description, Reflections involve capturing the client's communication and returning it, either explicitly or implicitly. The counselor's response here is: "I can hear that this shift to working from home has been really challenging for you and it's leading to feelings of depression. Can you tell me more about your experience and how it has been affecting you?"

Starting with step 1: Identify any reflections. The first part of the counselor's response, "I can hear that this shift to working from home has been really challenging for you and it's leading to feelings of depression," seems to rephrase the seeker's statement. The seeker mentioned feeling depressed from working from home post-pandemic. The counselor restates that the shift is challenging and causing depression. That's a reflection. The second part is an 

In [22]:
# Parse prediction
prediction, confidence = qwen_evaluator._parse_prediction(response, "Reflections-goodareas")
print(prediction)
print(confidence)

1
0.8


In [ ]:
eval_metrics = qwen_evaluator.evaluate_dataset(dataset=split_dataset, skill_class="Reflections-goodareas")

Evaluating Reflections-goodareas:   0%|▏                                                                                                     | 1/779 [01:07<14:37:37, 67.68s/it]

## Making predictions on Test/Study Data

In [41]:
import pandas as pd

# condition = "control"
# condition = "treatment"
# input_data = pd.read_csv(f"./all_{condition}_seekerhelper_pairs.csv")
input_data = pd.read_csv("N94_all_seekerhelper_pairs.csv")
print(len(input_data))
input_data.head()

3842


,id,seeker_post,response_post
0,14_a1df3c7155d0438b9c4084b57b66c6e6_0_0,NaN,good evening I understand you're feeling isola...
1,14_a1df3c7155d0438b9c4084b57b66c6e6_0_1,"Yeah, it's just... everyone was with their fam...",I hear you. It can be difficult growing apart ...
2,14_a1df3c7155d0438b9c4084b57b66c6e6_0_2,"Yeah, it's tough. You know, during the holiday...",Perhaps they're feeling similarly. Waiting for...
3,14_a1df3c7155d0438b9c4084b57b66c6e6_0_3,But why should I always have to be the bigger ...,"Being the bigger person can feel burdensome, c..."
4,14_a1df3c7155d0438b9c4084b57b66c6e6_0_4,"Yeah, exactly. It's like, why should I keep pu...",How would you like for them to show you they c...


In [42]:
from datasets import Dataset

study_dataset = Dataset.from_pandas(input_data)

wandb: ERROR Problem finishing run


In [43]:
from transformers import pipeline

# WHICH_CLASS="Reflections-goodareas"
WHICH_CLASS="Questions-goodareas"
# load model from huggingface.co/models using our repository id
# classifier = pipeline("sentiment-analysis", model=f"./roberta-Reflections-goodareas-eval_FeedbackESConv5pp_CARE10pp-sweeps-best-d6x1jzik-1741277930", device=0)
classifier = pipeline("sentiment-analysis", model=f"./roberta-Questions-goodareas-eval_FeedbackESConv5pp_CARE10pp-sweeps-best-82jc07j0-1741329550", device=0)

# classifier = pipeline("sentiment-analysis", model="ModernBERT-Empathy-goodareas-classifier", device=0)

# sample = f"Seeker: {input_data.loc[0, "seeker_post"]}\nHelper: {input_data.loc[0, "response_post"]}"
# pred = classifier(sample)
# print(pred)

def binary_prediction_seeker_response_post(seeker, helper):
    # sample = f"Seeker: {seeker}\nHelper: {helper}"
    sample = f"Seeker: {seeker}[SEP]Helper: {helper}"
    
    pred = classifier(sample)
    return int(pred[0]['label'] == 'selected')

Device set to use cuda:0


In [44]:
# output_preds = input_data.apply(binary_prediction_seeker_response_post, axis=0)

def predict_reflection(example):
    # Apply your binary prediction function to each example
    example["prediction"] = binary_prediction_seeker_response_post(
        example["seeker_post"], 
        example["response_post"]
    )
    return example

# Apply the function to the entire dataset at once
predicted_dataset = study_dataset.map(predict_reflection)

# strengths = [binary_prediction_seeker_response_post(input_data.loc[i, "seeker_post"], input_data.loc[i, "response_post"])
#              for i in range(len(input_data))]

Map: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3842/3842 [00:38<00:00, 99.94 examples/s]


In [45]:
input_data[f"{WHICH_CLASS}"] = predicted_dataset['prediction']
# input_data[f"{WHICH_CLASS}"] = strengths
# input_data["Empathy-goodareas"] = strengths

In [46]:
print(len(input_data))
input_data.head()

3842


,id,seeker_post,response_post,Questions-goodareas
0,14_a1df3c7155d0438b9c4084b57b66c6e6_0_0,NaN,good evening I understand you're feeling isola...,0
1,14_a1df3c7155d0438b9c4084b57b66c6e6_0_1,"Yeah, it's just... everyone was with their fam...",I hear you. It can be difficult growing apart ...,0
2,14_a1df3c7155d0438b9c4084b57b66c6e6_0_2,"Yeah, it's tough. You know, during the holiday...",Perhaps they're feeling similarly. Waiting for...,0
3,14_a1df3c7155d0438b9c4084b57b66c6e6_0_3,But why should I always have to be the bigger ...,"Being the bigger person can feel burdensome, c...",0
4,14_a1df3c7155d0438b9c4084b57b66c6e6_0_4,"Yeah, exactly. It's like, why should I keep pu...",How would you like for them to show you they c...,1


In [47]:
# input_data.to_csv(f'all_{condition}_seekerhelper_pairs_{WHICH_CLASS}.csv')
input_data.to_csv(f"N94_all_seekerhelper_pairs_{WHICH_CLASS}.csv")

In [37]:
f'all_{condition}_seekerhelper_pairs_{WHICH_CLASS}.csv'

'all_control_seekerhelper_pairs_Reflections-goodareas.csv'